# Restarting a simulation from a saved state

At the end of [../04_protein/01_villin_headpiece.ipynb](../04_protein/01_villin_headpiece.ipynb), the final state of the simulation was saved in three different ways:

- `final.pdb`: a PDB file with the positions and the periodic box.
- `final.xml`: an XML file with a serialized OpenMM `State`: positions, velocities, box, time and step count.
- `final.chk`: a binary checkpoint file, which also contains the internal state of the integrator and the random number generators.

The last cell of that notebook copies these files to this directory.
You need them here to run this notebook.

In this notebook, we start three new simulations of the villin headpiece,
each from a different file, and we check what is (and what is not) restored.
Because the checkpoint file is specific to the computer on which it was created,
run this notebook on the same computer as the villin notebook.

In [ ]:
from sys import stdout

import numpy as np
import openmm as mm
from openmm import app, unit

## 1. Rebuilding the simulation

None of the three files contains the force field parameters or the definition of the integrator and the barostat.
Before loading a saved state, you must therefore first create a `Simulation` object with exactly the same setup as in the villin notebook.
The PDB file `final.pdb` serves as the topology.
It already contains the hydrogen atoms, the water molecules and the periodic box, so no `Modeller` is needed.
The function below is a copy of the setup in the villin notebook.

In [ ]:
def build_simulation(topology):
    """Create a simulation with the same setup as in the villin notebook."""
    forcefield = app.ForceField("amber14-all.xml", "amber14/tip3pfb.xml")
    system = forcefield.createSystem(topology, nonbondedMethod=app.PME, constraints=app.HBonds)
    temperature = 300 * unit.kelvin
    pressure = 1 * unit.bar
    integrator = mm.LangevinMiddleIntegrator(
        temperature, 1 / unit.picosecond, 2 * unit.femtoseconds
    )
    system.addForce(mm.MonteCarloBarostat(pressure, temperature))
    return app.Simulation(topology, system, integrator)


pdb = app.PDBFile("final.pdb")

To see what each file restores, we compare the state of each restarted simulation with a reference.
The reference is the `State` object stored in `final.xml`, which is loaded here without creating a simulation.
Positions and velocities are converted to NumPy arrays in nm and nm/ps, so they can be compared directly.

In [ ]:
with open("final.xml") as f:
    reference = mm.XmlSerializer.deserialize(f.read())
ref_pos = reference.getPositions(asNumpy=True).value_in_unit(unit.nanometer)
ref_vel = reference.getVelocities(asNumpy=True).value_in_unit(unit.nanometer / unit.picosecond)
ref_box = reference.getPeriodicBoxVectors(asNumpy=True).value_in_unit(unit.nanometer)


def compare_with_reference(simulation):
    """Print how much the current state of a simulation differs from the reference state."""
    state = simulation.context.getState(getPositions=True, getVelocities=True)
    pos = state.getPositions(asNumpy=True).value_in_unit(unit.nanometer)
    vel = state.getVelocities(asNumpy=True).value_in_unit(unit.nanometer / unit.picosecond)
    box = state.getPeriodicBoxVectors(asNumpy=True).value_in_unit(unit.nanometer)
    print(f"Largest position difference: {abs(pos - ref_pos).max():.1e} nm")
    print(f"RMS velocity difference:     {np.sqrt(((vel - ref_vel) ** 2).mean()):.1e} nm/ps")
    print(f"Largest box difference:      {abs(box - ref_box).max():.1e} nm")
    print(f"Step count: {simulation.currentStep}, time: {state.getTime()}")

## 2. Restart from the PDB file

A PDB file contains no velocities.
Just like at the start of the first simulations, the velocities are therefore assigned randomly from the Maxwell-Boltzmann distribution.
The box is read from the `CRYST1` line and used by the `Simulation` through the topology.

In [ ]:
simulation_pdb = build_simulation(pdb.topology)
simulation_pdb.context.setPositions(pdb.positions)
simulation_pdb.context.setVelocitiesToTemperature(300 * unit.kelvin)
compare_with_reference(simulation_pdb)

The positions and the box deviate only by the rounding error of the PDB format, which stores coordinates in Å with three decimals ($10^{-4}\,\mathrm{nm}$).
The velocities are completely different, because they are random numbers.
The step counter and the time have also been reset to zero, because they are not stored in a PDB file.
A restart from a PDB file is not a continuation of the previous simulation, but it is a valid new simulation that starts from an equilibrated structure.
This is also the only option when the saved state comes from a different program.

## 3. Restart from the XML state

`loadState` puts the positions, velocities, box vectors, time and step count back into the simulation.

In [ ]:
simulation_xml = build_simulation(pdb.topology)
simulation_xml.loadState("final.xml")
compare_with_reference(simulation_xml)

## 4. Restart from the checkpoint

`loadCheckpoint` restores the same information, and also the internal state of the integrator and the random number generators.

In [ ]:
simulation_chk = build_simulation(pdb.topology)
simulation_chk.loadCheckpoint("final.chk")
compare_with_reference(simulation_chk)

Both the XML state and the checkpoint restore positions, velocities and box without loss of precision,
and also the step count and time, so the simulation can be continued at the point where it stopped.
Their difference only shows up in the continuation of the simulation, as demonstrated next.

## 5. How exact is the continuation?

The Langevin thermostat and the Monte Carlo barostat use random numbers.
The following function loads a saved state twice into the same simulation, runs 100 steps each time,
and reports the largest difference between the two final geometries.

In [ ]:
def continuation_difference(simulation, load, nsteps=100):
    """Load a saved state twice, run a few steps each time and compare the final positions."""
    final_positions = []
    for _ in range(2):
        load()
        simulation.step(nsteps)
        state = simulation.context.getState(getPositions=True)
        final_positions.append(state.getPositions(asNumpy=True).value_in_unit(unit.nanometer))
    return abs(final_positions[0] - final_positions[1]).max()


diff_xml = continuation_difference(simulation_xml, lambda: simulation_xml.loadState("final.xml"))
print(f"XML state:  largest position difference after 100 steps: {diff_xml:.1e} nm")
diff_chk = continuation_difference(
    simulation_chk, lambda: simulation_chk.loadCheckpoint("final.chk")
)
print(f"Checkpoint: largest position difference after 100 steps: {diff_chk:.1e} nm")

After loading the XML state, the thermostat and barostat continue with new random numbers, so the two continuations differ.
Both are valid samples of the same ensemble, but they are not identical.
After loading the checkpoint, the random number generators are also restored, so the two continuations stay much closer to each other (orders of magnitude, compared to the XML state).
They are not exactly identical, because of the finite precision of the arithmetic, e.g. the order in which forces are summed in parallel.
OpenMM does not guarantee that a checkpoint continuation is exactly the same as an uninterrupted simulation, and you should not rely on it.
In most cases, the XML state is sufficient, and it is more portable.

## 6. Continuing the simulation

Reporters are not stored in any of the files.
After a restart, you must attach them again.
The following cell continues the simulation from the checkpoint, and prints the step and time every 250 steps.
Note that the counters continue from the saved state, instead of starting again at zero.

In [ ]:
simulation_chk.loadCheckpoint("final.chk")
simulation_chk.reporters = []
simulation_chk.reporters.append(
    app.StateDataReporter(stdout, 250, step=True, time=True, temperature=True, potentialEnergy=True)
)
simulation_chk.step(1000)

When you continue a simulation that writes a DCD trajectory, you can extend the existing trajectory file
with the option `append=True`, e.g. `app.DCDReporter("traj.dcd", 10, append=True)`.
Make sure that the existing trajectory ends exactly at the step of the saved state.
Otherwise, there will be gaps or duplicated frames.
Also note that the simulation will not know about the original number of steps, so keep track of the remaining steps yourself.

**<span style="color:#A03;font-size:14pt">
&#x1F450; HANDS-ON! &#x1F528;
</span>**

> - Create a `Simulation` on the `Reference` platform, which is available on any computer:
>   `app.Simulation(topology, system, integrator, mm.Platform.getPlatformByName("Reference"))`.
>   Which of the files `final.xml` and `final.chk` can be loaded into this simulation, and which not?
>   Modify `build_simulation` to accept a platform argument.
> - Add a `CheckpointReporter` to the simulation of the villin notebook, with a checkpoint file `checkpoint.chk` that is overwritten every 1000 steps.
>   Interrupt the kernel while the simulation is running, and then restart from `checkpoint.chk`.
>   How can you find out how many steps still remain to reach a total of 30000 steps?